<h1> כריית נתונים בפייתון <br><br> חלק א - הבאת נתונים</h1>
<p> <br>https://github.com/Aviyashap/Python-based-Data-Mining-Study</p>

<h2> ייבוא ספריות </h2>

In [1]:
# במידת הצורך להריץ פעם אחת בלבד
#!pip install beautifulsoup4

In [2]:
import pandas as pd
import numpy as np
import json
import requests
import os
import re
import gc
from bs4 import BeautifulSoup

<h2> שימוש בטבלאות נתונים מתוך IMDb </h2>

<h3>טבלה title.basics.tsv.gz</h3>
עמודות הטבלה הרלוונטיות למטלה
<p>
<br>tconst (string) - קוד ייחודי לכל סרט/סדרה
<br>primaryTitle (string) – השם הפופולרי של הסרט
זמן ריצה
<br> startYear - שנת יציאה
<br> runtimeMinutes - זמן בדקות
<br>genres (string array) – כולל עד שלושה ז'אנרים לכל סרט </p>

In [3]:
#הבאת טבלת הסרטים הכללית
file_title_basics = 'https://datasets.imdbws.com/title.basics.tsv.gz'
needed_cols = ['tconst', 'primaryTitle', 'startYear', 'runtimeMinutes', 'genres', 'titleType']
df_title_basics = pd.read_csv(file_title_basics, sep='\t', usecols=needed_cols, na_values='\\N', compression='gzip')
print(df_title_basics.head())

C:\Users\Aviya Shapira\AppData\Local\Temp\ipykernel_11504\2695949991.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_title_basics = pd.read_csv(file_title_basics, sep='\t', usecols=needed_cols, na_values='\\N', compression='gzip')


      tconst titleType            primaryTitle  startYear runtimeMinutes  \
0  tt0000001     short              Carmencita     1894.0            1.0   
1  tt0000002     short  Le clown et ses chiens     1892.0            5.0   
2  tt0000003     short            Poor Pierrot     1892.0            5.0   
3  tt0000004     short             Un bon bock     1892.0           12.0   
4  tt0000005     short        Blacksmith Scene     1893.0            1.0   

                     genres  
0         Documentary,Short  
1           Animation,Short  
2  Animation,Comedy,Romance  
3           Animation,Short  
4                     Short  


In [4]:
#סינון לסרטים בלבד
df_title_basics_movies = df_title_basics[df_title_basics['titleType'] == 'movie'].copy()
print(df_title_basics_movies['titleType'].unique())

['movie']


In [5]:
#סינון לפי אותיות G+H
G_H = df_title_basics_movies['primaryTitle'].str.startswith(('G', 'H', 'g', 'h'), na=False)
df_title_basics_movies_G_H = df_title_basics_movies[G_H].copy()
print(df_title_basics_movies_G_H['primaryTitle'])

625                                        Hamlet
869         Gira política de Madero y Pino Suárez
876                     Hamlet, Prince of Denmark
1037                               Gøngehøvdingen
1100                                       Hamlet
                            ...                  
12476025                            Hayatta Olmaz
12476240                               Hellavator
12476424                               Hellavator
12477610                    Hiniku: Nure nawazeme
12477968                             However Long
Name: primaryTitle, Length: 47048, dtype: object


In [6]:
# זמן סרט + סינון לפי שנת יציאה
df_title_basics_movies_G_H['startYear'] = pd.to_numeric(df_title_basics_movies_G_H['startYear'], errors='coerce')
df_title_basics_movies_G_H['runtimeMinutes'] = pd.to_numeric(df_title_basics_movies_G_H['runtimeMinutes'], errors='coerce')
df_final = df_title_basics_movies_G_H[df_title_basics_movies_G_H['startYear'] <= 2024]
df_final = df_final[(df_final['runtimeMinutes'] < 300) & (df_final['runtimeMinutes'] > 60)]

print(df_final.head())
len (df_final)

#מחיקת הטבלאות הגדולות לניקוי הזיכרון
del df_title_basics
del df_title_basics_movies
gc.collect()

         tconst titleType                     primaryTitle  startYear  \
2871  tt0002898     movie  Germinal; or, The Toll of Labor     1913.0   
2895  tt0002922     movie                           Hamlet     1913.0   
4003  tt0004047     movie                       Half Breed     1913.0   
6630  tt0006715     movie               Genie tegen geweld     1916.0   
6643  tt0006728     movie      God's Country and the Woman     1916.0   

      runtimeMinutes               genres  
2871           150.0                Drama  
2895            64.0                Drama  
4003            69.0                Drama  
6630            70.0  Crime,Drama,Mystery  
6643            80.0        Drama,Romance  


19

<h3>טבלה title.ratings.tsv.gz</h3>
עמודות הטבלה הרלוונטיות למטלה
<p>
<br>tconst (string) - קוד ייחודי לכל סרט/סדרה
<br>AverageRating- ממוצע צפיות
<br> numVotes- מספר הצבעות 
 </p>

In [7]:
file_title_ratings = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
df_title_ratings = pd.read_csv(file_title_ratings, sep='\t', na_values='\\N', compression='gzip')
print(df_title_ratings.head())

      tconst  averageRating  numVotes
0  tt0000001            5.7      2213
1  tt0000002            5.5       317
2  tt0000003            6.4      2327
3  tt0000004            5.1       199
4  tt0000005            6.2      3047


In [8]:
#איחוד עם הטבלה הקודמת רק של השורות הרלוונטיות
df_combined = pd.merge(df_final, df_title_ratings, on='tconst', how='inner')
print(df_combined.columns)

Index(['tconst', 'titleType', 'primaryTitle', 'startYear', 'runtimeMinutes',
       'genres', 'averageRating', 'numVotes'],
      dtype='object')


<h3>טבלה title.principals.tsv.gz</h3>
עמודות הטבלה הרלוונטיות למטלה
<p>
<br>tconst (string) - קוד ייחודי לכל סרט/סדרה
<br>nconst- קוד ייחודי לשחקן
<br> characters- שחקנים  
 </p>

In [9]:
file_title_principals = 'https://datasets.imdbws.com/title.principals.tsv.gz'
needed_cols = ['tconst', 'nconst', 'category', 'ordering']

dtype_settings = {
    'tconst': 'str',
    'nconst': 'str',
    'category': 'category', 
    'ordering': 'int8'
}

# שימוש ב-iterator כדי לקרוא את הקובץ ב"מנות" קטנות ולא בבת אחת
chunks = pd.read_csv(file_title_principals, 
                     sep='\t', 
                     usecols=needed_cols, 
                     dtype=dtype_settings,
                     na_values='\\N', 
                     chunksize=500000)


# חיבור הכל לטבלה
title_principals = pd.concat(chunks)

# ניקוי
del chunks
gc.collect()

print(title_principals.head())

      tconst  ordering     nconst         category
0  tt0000001         1  nm1588970             self
1  tt0000001         2  nm0005690         director
2  tt0000001         3  nm0005690         producer
3  tt0000001         4  nm0374658  cinematographer
4  tt0000002         1  nm0721526         director


In [10]:
#סינון לחמשת השחקנים המובילים
df_lead_actors = title_principals[(title_principals['category'].isin(['actor', 'actress'])) & (title_principals['ordering'] <= 5)].copy()

In [11]:
#איחוד הטבלאות
df_lead_actors = df_lead_actors[['tconst', 'nconst', 'ordering']]
df_lead_actors = df_lead_actors.sort_values(by=['tconst', 'ordering'])
df_actors_grouped = df_lead_actors.groupby('tconst')['nconst'].apply(lambda x: ', '.join(x)).reset_index()

df_combined = pd.merge(df_combined, df_actors_grouped, on='tconst', how='left')
print(df_combined.head(10))

#מחיקת טבלאות כבדות
del df_actors_grouped
del df_lead_actors
gc.collect()

      tconst titleType                                    primaryTitle  \
0  tt0002898     movie                 Germinal; or, The Toll of Labor   
1  tt0002922     movie                                          Hamlet   
2  tt0004047     movie                                      Half Breed   
3  tt0006715     movie                              Genie tegen geweld   
4  tt0006780     movie                                   Hell's Hinges   
5  tt0006809     movie                       His Picture in the Papers   
6  tt0006819     movie                           Hoffmanns Erzählungen   
7  tt0006820     movie                             Homunculus, 1. Teil   
8  tt0006826     movie                                      Hoodoo Ann   
9  tt0007240     movie  Homunculus, 4. Teil - Die Rache des Homunculus   

   startYear  runtimeMinutes                 genres  averageRating  numVotes  \
0     1913.0           150.0                  Drama            7.0       197   
1     1913.0            6

0

In [12]:
len(df_combined)

17250

<h2>web scraping הבאת הנתונים החסרים מויקפדיה</h2>

In [13]:
#פונקציה להבאת מידע בודד עבור סרט אחד
def get_movie_info(title, year):
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    # מילון ברירת מחדל אם לא יימצא כלום
    data = {'Language': None, 'Country': None, 'Budget': None, 'Box Office': None, 'Plot': None}
    
    # ניקוי שנה ושם לפורמט URL
    try:
        y_str = str(int(float(year))) if year else ""
    except:
        y_str = ""
    t_under = title.replace(' ', '_')
    
    urls = [
        f"https://en.wikipedia.org/wiki/{t_under}_({y_str}_film)",
        f"https://en.wikipedia.org/wiki/{t_under}_(film)",
        f"https://en.wikipedia.org/wiki/{t_under}",
        f"https://en.wikipedia.org/wiki/{t_under}_({y_str})"
    ]
    
    valus = None
    for url in urls:
        try:
            request_data = requests.get(url, headers=headers, timeout=5)
            if request_data.status_code == 200:
                valus = request_data
                break
        except:
            continue
            
    if not valus: 
        return data

    soup = BeautifulSoup(valus.text, 'html.parser')

    #הבאת המידע הנמצא בטבלה בויקפדיה
    info = soup.find("table", class_="infobox")
    if info:
        for tr in info.find_all("tr"):
            th, td = tr.find("th"), tr.find("td")
            if th and td:
                lbl = th.get_text().lower()
                val = td.get_text(" ").strip()
                if "language" in lbl: data['Language'] = val
                elif "country" in lbl: data['Country'] = val
                elif "budget" in lbl: data['Budget'] = val
                elif "box office" in lbl: data['Box Office'] = val

    #הבאת המידע של התקציר שנמצא בתוכן הכללי של הדף
    elements = soup.find_all(['h2', 'p'])
    title, paragraphs = False, []
    for x in elements:
        if x.name == 'h2':
            txt = x.get_text().lower()
            if any(w in txt for w in ['plot', 'synopsis', 'summary']):
                title = True
                continue
            if title: break
        if title and x.name == 'p':
            paragraphs.append(x.get_text().strip())

    data['Plot'] = " ".join(paragraphs)
    return data

In [14]:
# בדיקה על סרט אחד
print(get_movie_info("His Picture in the Papers", 1916.0))

{'Language': 'Silent  (English  intertitles )', 'Country': 'United States', 'Budget': '$42,599.94 [ 1 ]', 'Box Office': None, 'Plot': "Pete Prindle, son of Proteus, is a vegetarian health food manufacturer who wishes to marry Christine Cadwalader. She agrees. However, Proteus considers his son lazy, with no contributions to the company and therefore undeserving of his father's wealth. His daughters have their pictures in the newspapers, pictures of them promoting the company products. Cassius refuses to consent to his daughter's hand since he believes Pete to be lazy as well, with no real stake in his father's company. Pete tries hard to get in the newspaper: He fakes a car accident, which gets an insignificant mention in the paper. He wins a boxing match, which turns out to be an illegally run ring which ends up being raided by police. After a misunderstanding, he washes up on the shore in his pajamas after falling off a cruise ship, and proceeds to beat two police officers, his name 

In [15]:
#get_movie_infoפונקציה ליצירת מילונים עבור כל מידע שמתקבל מפונקציית ה
def get_all_movies_data(dataframe):
    results = []
    
    #לולאה שרצה על הטבלה ומוציאה נותני ויקפדיה לכל סרט
    for index, row in dataframe.iterrows():
        
        # קריאה לפונקציה הראשונה וקבלת המילון עם הנתונים מויקיפדיה
        wiki_data = get_movie_info(row['primaryTitle'], row['startYear'])
        
        #מילון חדש עם הנתונים שמצאנו
        movie_dict = {}
        movie_dict['name'] = row['primaryTitle']
        movie_dict['year'] = row['startYear']
        movie_dict['language'] = wiki_data.get('Language')
        movie_dict['country'] = wiki_data.get('Country')
        movie_dict['budget'] = wiki_data.get('Budget')
        movie_dict['box_office'] = wiki_data.get('Box Office')
        movie_dict['plot'] = wiki_data.get('Plot')
        
        # הוספת המילון לרשימה הכללית
        results.append(movie_dict)
            
    return results

In [16]:
# דוגמה על 20 הסרטים הראשונים
movies_list = get_all_movies_data(df_combined.head(20))
pd.DataFrame(movies_list)

,name,year,language,country,budget,box_office,plot
0,"Germinal; or, The Toll of Labor",1913.0,None,None,None,None,None
1,Hamlet,1913.0,None,United Kingdom,None,None,"After the Ghost of his father appears, Hamlet ..."
2,Half Breed,1913.0,Silent Swedish intertitles,Sweden,None,None,
3,Genie tegen geweld,1916.0,Silent,Netherlands,None,None,
4,Hell's Hinges,1916.0,Silent English intertitles,United States,None,None,Hell's Hinges tells the story of a weak-willed...
5,His Picture in the Papers,1916.0,Silent (English intertitles ),United States,"$42,599.94 [ 1 ]",None,"Pete Prindle, son of Proteus, is a vegetarian ..."
6,Hoffmanns Erzählungen,1916.0,French,None,None,None,A tavern in Nuremberg: The Muse appears and re...
7,"Homunculus, 1. Teil",1916.0,None,None,None,None,None
8,Hoodoo Ann,1916.0,Silent film English intertitles,United States,None,None,Ann (Mae Marsh) is a young girl who has lived ...
9,"Homunculus, 4. Teil - Die Rache des Homunculus",1917.0,None,None,None,None,None


In [17]:
#קוד עם לולאה מפוצלת למקטעים להבאת כלל הנתונים מויקפדיה עבור כל הסרטים

all_results = []  # הרשימה הסופית שתכיל את כל המילונים
total_movies = len(df_combined) # מספר הסרטים הכולל
chunk_size = 100 # גודל כל מקטע

# לולאה שרצה במקטעים
# range(start, stop, step)
for i in range(0, total_movies, chunk_size):
    # חותכים את המקטע הנוכחי מה-DataFrame
    current_chunk = df_combined.iloc[i : i + chunk_size]
    
    print(f"Starting chunk: movies {i} to {i + chunk_size}...")
    
    #עבור כל סרט בקבוצה הנוכחית שימוש בפונקציה שמחזירה רשימת מילונים
    chunk_list = get_all_movies_data(current_chunk)
    
    # הוספת המילונים מהמקטע הנוכחי לרשימה הגדולה
    all_results.extend(chunk_list)
    
print("Finished processing all movies!")

# בסיום, הופכים את כל הרשימה הגדולה ל-DataFrame 
df_wiki_final = pd.DataFrame(all_results)

Starting chunk: movies 0 to 100...
Starting chunk: movies 100 to 200...
Starting chunk: movies 200 to 300...
Starting chunk: movies 300 to 400...
Starting chunk: movies 400 to 500...
Starting chunk: movies 500 to 600...
Starting chunk: movies 600 to 700...
Starting chunk: movies 700 to 800...
Starting chunk: movies 800 to 900...
Starting chunk: movies 900 to 1000...
Starting chunk: movies 1000 to 1100...
Starting chunk: movies 1100 to 1200...
Starting chunk: movies 1200 to 1300...
Starting chunk: movies 1300 to 1400...
Starting chunk: movies 1400 to 1500...
Starting chunk: movies 1500 to 1600...
Starting chunk: movies 1600 to 1700...
Starting chunk: movies 1700 to 1800...
Starting chunk: movies 1800 to 1900...
Starting chunk: movies 1900 to 2000...
Starting chunk: movies 2000 to 2100...
Starting chunk: movies 2100 to 2200...
Starting chunk: movies 2200 to 2300...
Starting chunk: movies 2300 to 2400...
Starting chunk: movies 2400 to 2500...
Starting chunk: movies 2500 to 2600...
Startin

<h2>סידור המידע מויקפדיה</h2>

In [127]:
#עותק של המידע מויקפדיה שלא אאבד מידע
df_wiki_back = pd.DataFrame(all_results)
df_wiki_back.to_csv("df_wiki.csv", index=False, encoding='utf-8-sig')

In [299]:
df_wiki_clean = df_wiki_back.copy()
df_wiki_clean.head(15)

,name,year,language,country,budget,box_office,plot
0,"Germinal; or, The Toll of Labor",1913.0,None,None,None,None,None
1,Hamlet,1913.0,None,United Kingdom,None,None,"After the Ghost of his father appears, Hamlet ..."
2,Half Breed,1913.0,Silent Swedish intertitles,Sweden,None,None,
3,Genie tegen geweld,1916.0,Silent,Netherlands,None,None,
4,Hell's Hinges,1916.0,Silent English intertitles,United States,None,None,Hell's Hinges tells the story of a weak-willed...
5,His Picture in the Papers,1916.0,Silent (English intertitles ),United States,"$42,599.94 [ 1 ]",None,"Pete Prindle, son of Proteus, is a vegetarian ..."
6,Hoffmanns Erzählungen,1916.0,French,None,None,None,A tavern in Nuremberg: The Muse appears and re...
7,"Homunculus, 1. Teil",1916.0,None,None,None,None,None
8,Hoodoo Ann,1916.0,Silent film English intertitles,United States,None,None,Ann (Mae Marsh) is a young girl who has lived ...
9,"Homunculus, 4. Teil - Die Rache des Homunculus",1917.0,None,None,None,None,None


In [300]:
#פונקציה להמרה למיליוני דולרים
def cleaner(value):
    if pd.isna(value) or str(value).strip().lower() in ['none', '', 'nan', 'unknown', 'tba']:
        return None
    
    # 1. ניקוי יסודי
    text = str(value).lower().strip()
    text = re.sub(r'\[.*?\]', '', text) 
    text = text.replace('~', '').replace('\xa0', ' ').replace(',', '')
    text = " ".join(text.split())

    # 2. עדיפות לדולר ($) - טיפול מהיר וסופי
    usd_match = re.search(r'\$\s*(\d+\.?\d*)\s*(million|billion|m|b)?', text)
    if usd_match:
        num = float(usd_match.group(1))
        unit = usd_match.group(2)
        if unit in ['billion', 'b']: return round(num * 1000, 3)
        if unit in ['million', 'm']: return round(num, 3)
        return round(num / 1_000_000, 3)

    # 3. מפת שערי חליפין
    rates = {
        'toman': 0.000024, '₤': 0.0006, 'lira': 0.0006,
        '₩': 0.00075, 'won': 0.00075, 'krw': 0.00075,
        '₹': 0.012, 'rs': 0.012, 'inr': 0.012,
        '€': 1.10, '£': 1.25, 'vnd': 0.00004, 'dong': 0.00004,
        'rp': 0.000065, '₱': 0.018, 'il': 0.06
    }
    
    current_rate = 1.0
    matched_currency = False
    for sym, r in sorted(rates.items(), key=len, reverse=True):
        if sym in text:
            current_rate = r
            matched_currency = True
            break
    
    if not matched_currency and any(x in text for x in ['crore', 'lakh', 'cr']):
        current_rate = 0.012

    # 4. פונקציית עזר חכמה לזיהוי ערך של מספר בודד בתוך טקסט
    def get_single_value(num_str, full_context, local_context=""):
        n = float(num_str)
        # בדיקה אם ליד המספר הספציפי יש מילת מכפיל
        combined_context = (full_context + " " + local_context).lower()
        
        if 'billion' in combined_context or ' b ' in combined_context: return n * 1_000_000_000
        if 'crore' in combined_context or 'cr' in combined_context: return n * 10_000_000
        if 'million' in combined_context or ' m ' in combined_context: return n * 1_000_000
        if 'lakh' in combined_context: return n * 100_000
        
        # אם אין מכפיל והמספר קטן מ-1000, נניח שהכוונה למיליונים (כמו "1.5 - 2")
        if n < 1000: return n * 1_000_000
        
        # אם המספר גדול (כמו 650,000), הוא ערך גולמי
        return n

    # 5. טיפול בטווחים (מזהה כל צד בנפרד)
    range_match = re.search(r'(\d+\.?\d*)\s*[-–]\s*(\d+\.?\d*)', text)
    if range_match:
        n1_str, n2_str = range_match.group(1), range_match.group(2)
        
        # נבדוק אם המילה million מופיעה רק בסוף המשפט או גם ליד המספר הראשון
        v1 = get_single_value(n1_str, text if 'million' in text and float(n1_str) < 1000 else n1_str)
        v2 = get_single_value(n2_str, text)
        full_val = (v1 + v2) / 2
    else:
        # טיפול בנקודה אירופאית
        if '.' in text and len(text.split('.')[-1]) == 3 and 'million' not in text:
            text = text.replace('.', '')
        
        nums = re.findall(r'(\d+\.?\d*)', text)
        if not nums: return None
        full_val = get_single_value(nums[0], text)

    # 6. חישוב סופי
    final_usd = (full_val * current_rate) / 1_000_000
    
    # הגנה נגד חריגות במטבעות חלשים
    if final_usd > 3000 and current_rate < 0.001:
        final_usd /= 1000

    return round(final_usd, 3)

In [301]:
# הפעלה על הנתונים
df_wiki_clean['budget'] = df_wiki_clean['budget'].apply(cleaner)
df_wiki_clean['box_office'] = df_wiki_clean['box_office'].apply(cleaner)

#בדיקה
df_wiki_clean.sample(15)

,name,year,language,country,budget,box_office,plot
6539,Guess Who,2005.0,English,United States,35.0,103.000,Theresa Jones takes her boyfriend Simon Green ...
7904,Hello Jindagi,2022.0,None,None,NaN,NaN,None
6273,Hot Club California,1999.0,None,None,NaN,NaN,None
7636,Hammer & Tickle,2006.0,English,United States,NaN,NaN,
10460,Head Cold,2010.0,None,None,NaN,NaN,
2085,"Good to See You Again, Alice Cooper",1974.0,None,None,NaN,NaN,
10617,Guilt & Sentence,2010.0,None,None,NaN,NaN,None
1202,"Good-bye, My Lady",1956.0,English,United States,NaN,NaN,Young orphan boy Skeeter (Brandon deWilde) is ...
13753,Homegrown,2024.0,English,United States,NaN,NaN,
13089,Glowzies,2023.0,None,None,NaN,NaN,None


In [302]:
#פונקציה להגדרת השפה הראשית בלבד
def clean_language_final(value):
    if value is None or pd.isna(value) or value == 'None':
        return None
    
    # המרה לטקסט וניקוי רווחים קיצוניים
    val = str(value).strip()
    
    #רק את מה שמופיע לפני המפריד הראשון
    delimiters = [',', ';', '/', ' and ']
    for d in delimiters:
        val = val.split(d)[0]
    
    # הביטוי הזה מוצא את רצף האותיות הראשון (באנגלית או עברית)
    match = re.search(r'([a-zA-Zא-ת]+)', val)
    
    if match:
        result = match.group(1).strip().capitalize()
        # בדיקה קטנה: אם המילה קצרה מדי (פחות מ-2 אותיות), כנראה זה זבל
        return result if len(result) > 1 else None
        
    return None

# החלה על ה-DataFrame
df_wiki_clean['language'] = df_wiki_clean['language'].apply(clean_language_final)
print (df_wiki_clean['language'].unique())

[None 'Silent' 'French' 'Sound' 'English' 'German' 'Russian' 'Danish'
 'Hungarian' 'Polish' 'Swedish' 'Italian' 'Spanish' 'Portuguese' 'Dutch'
 'Yiddish' 'Japanese' 'Norwegian' 'Swiss' 'Cantonese' 'Hindi' 'Early'
 'Finnish' 'Serbo' 'Bengali' 'Italy' 'Czech' 'Chinese' 'Hebrew' 'Mandarin'
 'Kannada' 'Croatian' 'Burmese' 'Persian' 'Amharic' 'Arabic' 'Korean'
 'Armenian' 'Scottish' 'Romanian' 'Assamese' 'Telugu' 'Wolof' 'Welsh'
 'Luxembourgish' 'Bambara' 'Taiwanese' 'Filipino' 'Greek' 'Turkish' 'Urdu'
 'Romani' 'Malayalam' 'Greenlandic' 'Marathi' 'Tamil' 'Koine' 'Estonian'
 'Bulgarian' 'Macedonian' 'Icelandic' 'Dolpo' 'Hindustani' 'Thai' 'Malay'
 'Slovak' 'Bangla' 'Tagalog' 'Odia' 'Egyptian' 'No' 'Punjabi' 'Slovene'
 'Sinhala' 'Hittite' 'Serbian' 'Bosnian' 'Indonesian' 'Bavarian' 'Sorani'
 'Neapolitan' 'Bhojpuri' 'Irish' 'Maltese' 'Gujarati' 'Maithili' 'Nepali'
 'Vietnamese' 'Dari' 'Uzbek' 'Tulu' 'Lao' 'Dhivehi' 'Moroccan' 'Avar'
 'Algerian' 'Albanian' 'Sudanese' 'Bahasa' 'Tunisian' 'Catal

<h2>איחוד המידע מכל המקורות לטבלה אחת</h2>

In [303]:
# איחוד שתי טבלאות המידע מויקפדיה וIMDB
df_final_1 = pd.merge(df_combined, df_wiki_clean, 
                     left_on=['primaryTitle', 'startYear'], 
                     right_on=['name', 'year'], 
                     how='left')


# הסרת עמודות עזר של האיחוד
df_final_1 = df_final_1.drop(columns=['name', 'year'])

if 'titleType' in df_final_1.columns:
    df_final_1 = df_final_1.drop(columns=['titleType'])

# שינוי שמות עמודות לפורמט הסופי
rename_dict = {
    'nconst': 'lead_actors_ids',
    'box_office': 'BoxOffice',
    'budget': 'Budget',
    'language': 'Language',
    'country': 'Country',
    'genres': 'Genres',
    'tconst': 'Tconst',
    'plot': 'Plot'
}
df_final_1 = df_final_1.rename(columns=rename_dict)


In [304]:
#הסרת כפיליות מחשש למידע מוטעה (הרחבה בדוח)
df_final_clean = df_final_1.drop_duplicates(subset=['primaryTitle', 'startYear'], keep=False)

In [305]:
#השארת נתונים כפולים רק על הסרט הפופולרי ביותר (הרחבה בדוח)

#סינון רק של השורות שיש בהן עלילה (כדי לא לספור NaN ככפולים)
df_with_plots = df_final_clean[df_final_clean['Plot'].notna()]

#מציאת כל השורות שבהן העלילה מופיעה יותר מפעם אחת
# keep=False גורם לזה שכל המופעים של העלילה הכפולה יופיעו (גם המקור וגם הכפיל)
duplicate_plots = df_with_plots[df_with_plots.duplicated(subset=['Plot'], keep=False)]

#הדפסת התוצאות
print(f"סהכ כל שורות עם עלילה משוכפלת: {len(duplicate_plots)}")
print(f"מספר העלילות הייחודיות שחזרו על עצמן: {duplicate_plots['Plot'].nunique()}")


סה כל שורות עם עלילה משוכפלת: 4622
מספר העלילות הייחודיות שחזרו על עצמן: 284


In [311]:
# מיון לפי מספר הצבעות כדי שהסרט ה"מרכזי" יהיה ראשון
df_final_clean = df_final_clean.sort_values('numVotes', ascending=False)

#יצירת מסכה (Mask) שמזהה את כל הכפילויות של העלילה (חוץ מהראשונה)
# שימי לב: אנחנו לא מוחקים את השורות, רק מזהים אותן
mask_duplicated_plots = df_final_clean.duplicated(subset=['Plot'], keep='first') & df_final_clean['Plot'].notna()

# רשימת כל העמודות שהגיעו מוויקיפדיה ואנחנו רוצים לאפס
columns_to_reset = ['Budget', 'BoxOffice', 'Plot', 'Language', 'Country']

#איפוס מוחלט של כל המידע ה"חשוד" כשגוי בשורות הללו
df_final_clean.loc[mask_duplicated_plots, columns_to_reset] = np.nan

# בדיקה כמה נשארו עם תקציב אמיתי עכשיו
print(f"סרטים שנשארו עם תקציב לאחר ניקוי כפילויות עלילה: {df_final_clean['Budget'].notna().sum()}")

סרטים שנשארו עם תקציב לאחר ניקוי כפילויות עלילה: 1494


In [312]:
# בדיקת ערכים חסרים
null_counts = df_final_clean.isnull().sum()
null_percentages = (df_final_clean.isnull().sum() / len(df_final_clean)) * 100
print(null_counts)
print(null_percentages)

# שמירת הקובץ הסופי של הנתונים
df_final_clean.to_csv("df_all_movies_final.csv", index=False, encoding='utf-8-sig')

print("הקובץ המאוחד והנקי נשמר בהצלחה")

Tconst                 0
primaryTitle           0
startYear              0
runtimeMinutes         0
Genres               353
averageRating          0
numVotes               0
lead_actors_ids     1767
Language           12209
Country            12739
Budget             15539
BoxOffice          15250
Plot               12040
dtype: int64
Tconst              0.000000
primaryTitle        0.000000
startYear           0.000000
runtimeMinutes      0.000000
Genres              2.072448
averageRating       0.000000
numVotes            0.000000
lead_actors_ids    10.373980
Language           71.678506
Country            74.790113
Budget             91.228791
BoxOffice          89.532085
Plot               70.686315
dtype: float64
הקובץ המאוחד והנקי נשמר בהצלחה


In [309]:
#Type של כל עמודה
print(df_final_clean.dtypes)

Tconst              object
primaryTitle        object
startYear          float64
runtimeMinutes     float64
Genres              object
averageRating      float64
numVotes             int64
lead_actors_ids     object
Language            object
Country             object
Budget             float64
BoxOffice          float64
Plot                object
dtype: object


In [310]:
len(df_final_clean)

17033